<a href="https://colab.research.google.com/github/jhstrong11/bootcamp-git-2025/blob/main/HW_4_Transformers_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OPT-125M Transformer Investigation

This notebook investigates Meta's OPT-125M decoder-only Transformer by comparing:

1. Formal transformer algorithms
2. Hugging Face's PyTorch OPT implementation
3. Runtime tensor dimensions

Model: facebook/opt-125m

In [2]:
!pip install -q transformers

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cpu


In [3]:
model_name = "facebook/opt-125m"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    attn_implementation="eager"
)

model.eval()

print("Model loaded successfully.")

config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  251MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Model loaded successfully.


In [4]:
config = model.config

print("Vocabulary size:", config.vocab_size)
print("Hidden size:", config.hidden_size)
print("Number of layers:", config.num_hidden_layers)
print("Number of attention heads:", config.num_attention_heads)
print("FFN dimension:", config.ffn_dim)
print("Maximum positions:", config.max_position_embeddings)

Vocabulary size: 50272
Hidden size: 768
Number of layers: 12
Number of attention heads: 12
FFN dimension: 3072
Maximum positions: 2048


## Notation

Throughout this investigation:

- \(B\) = batch size
- \(T\) = sequence length
- \(V\) = vocabulary size
- \(d_{\text{model}}\) = hidden size
- \(H\) = number of attention heads
- \(d_{\text{head}} = d_{\text{model}} / H\) = size of each attention head
- \(d_{\text{ff}}\) = feed-forward network dimension
- \(L\) = number of Transformer decoder layers
- \(T_{\max}\) = maximum supported sequence length

For OPT-125M:

- \(V = 50{,}272\)
- \(d_{\text{model}} = 768\)
- \(H = 12\)
- \(d_{\text{head}} = 64\)
- \(d_{\text{ff}} = 3072\)
- \(L = 12\)
- \(T_{\max} = 2048\)

All matrices are represented using row-major notation.

In [5]:
text = "Transformers learn relationships between tokens."

inputs = tokenizer(text, return_tensors="pt")

print("Tokens:", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))
print("Input IDs:", inputs["input_ids"])
print("Input IDs shape:", inputs["input_ids"].shape)
print("Attention mask shape:", inputs["attention_mask"].shape)

Tokens: ['</s>', 'Transform', 'ers', 'Ġlearn', 'Ġrelationships', 'Ġbetween', 'Ġtokens', '.']
Input IDs: tensor([[    2, 44820,   268,  1532,  4158,   227, 22121,     4]])
Input IDs shape: torch.Size([1, 8])
Attention mask shape: torch.Size([1, 8])


# Component 1: Input Representation

## 1. Algorithmic Specification

The input representation combines a learned token embedding with a learned positional embedding for each token position.

### Input

- Token IDs with shape `(B, T)`
- Attention mask with shape `(B, T)`

### Output

- Hidden-state representation with shape `(B, T, d_model)`

### Learned Parameters

- Token embedding matrix `E` with shape `(V, d_model)`
- Positional embedding matrix `P` with shape `(T_max + 2, d_model)`

### Relevant Model Settings

- `V` = vocabulary size
- `d_model` = hidden size
- `T_max` = maximum sequence length

## 2. Predicted Dimensions

For OPT-125M:

- Batch size: `B = 1`
- Sequence length: `T = 8`
- Vocabulary size: `V = 50,272`
- Hidden size: `d_model = 768`
- Maximum sequence length: `T_max = 2048`

Predicted dimensions:

- Token IDs: `(1, 8)`

- Token embedding matrix: `(V, d_model) = (50,272, 768)`

- Selected token embeddings: `(B, T, d_model) = (1, 8, 768)`

- Positional embedding matrix: `(T_max + 2, d_model) = (2050, 768)`

OPT uses an offset of 2 for its learned positional embeddings, so two additional rows are stored beyond the configured maximum sequence length.

- Selected positional embeddings: `(B, T, d_model) = (1, 8, 768)`

- Combined hidden representation: `(B, T, d_model) = (1, 8, 768)`

The selected token embeddings and positional embeddings must have the same dimensions so that they can be added together.

In [6]:
decoder = model.model.decoder

print("Token embedding layer:")
print(decoder.embed_tokens)

print("\nPositional embedding layer:")
print(decoder.embed_positions)

print("\nToken embedding weight shape:")
print(decoder.embed_tokens.weight.shape)

print("\nPositional embedding weight shape:")
print(decoder.embed_positions.weight.shape)

Token embedding layer:
Embedding(50272, 768, padding_idx=1)

Positional embedding layer:
OPTLearnedPositionalEmbedding(2050, 768)

Token embedding weight shape:
torch.Size([50272, 768])

Positional embedding weight shape:
torch.Size([2050, 768])


## 3. Formal Pseudocode

### Input

- Token IDs `X` with shape `(B, T)`
- Attention mask `M` with shape `(B, T)`

### Parameters

- Token embedding matrix `E` with shape `(V, d_model)`
- Positional embedding matrix `P` with shape `(T_max + 2, d_model)`

### Procedure

1. Look up the token embedding for each token ID:

`X_tok <- E[X]`

2. Determine the position index for each non-padding token.

3. Look up the learned positional embedding for each position:

`X_pos <- P[positions]`

4. Add the token and positional representations:

`H^(0) <- X_tok + X_pos`

### Output

`H^(0)` with shape `(B, T, d_model)`

In [8]:
decoder = model.model.decoder

token_embeddings = decoder.embed_tokens(inputs["input_ids"])
position_embeddings = decoder.embed_positions(inputs["attention_mask"])

hidden_states = token_embeddings + position_embeddings

print("Token embedding matrix:", decoder.embed_tokens.weight.shape)
print("Positional embedding matrix:", decoder.embed_positions.weight.shape)

print("Selected token embeddings:", token_embeddings.shape)
print("Selected positional embeddings:", position_embeddings.shape)
print("Combined hidden states:", hidden_states.shape)

Token embedding matrix: torch.Size([50272, 768])
Positional embedding matrix: torch.Size([2050, 768])
Selected token embeddings: torch.Size([1, 8, 768])
Selected positional embeddings: torch.Size([1, 8, 768])
Combined hidden states: torch.Size([1, 8, 768])


## 4. Observed Results and Corrections

The observed token embedding matrix had dimensions:

`(50,272, 768)`

The positional embedding matrix had dimensions:

`(2050, 768)`

The positional embedding table has two more rows than the configured maximum sequence length of 2048 because OPT uses a positional embedding offset of 2.

For the test sequence, the selected token embeddings had dimensions:

`(1, 8, 768)`

and the selected positional embeddings also had dimensions:

`(1, 8, 768)`

Because these tensors have matching dimensions, they can be added together. The resulting initial hidden representation had dimensions:

`(1, 8, 768)`

My initial expectation was that the positional embedding table would contain exactly `T_max = 2048` rows. After inspecting the OPT implementation and observing the tensor dimensions, I corrected this to:

`(T_max + 2, d_model) = (2050, 768)`

All other observed dimensions matched the predictions.

In [23]:
# Component 1 probes

batch_size, seq_len = inputs["input_ids"].shape
hidden_size = config.hidden_size
vocab_size = config.vocab_size
max_positions = config.max_position_embeddings

assert decoder.embed_tokens.weight.shape[0] == vocab_size
assert decoder.embed_tokens.weight.shape[1] == hidden_size

assert token_embeddings.shape == (
    batch_size,
    seq_len,
    hidden_size
)

assert position_embeddings.shape == (
    batch_size,
    seq_len,
    hidden_size
)

assert hidden_states.shape == (
    batch_size,
    seq_len,
    hidden_size
)

assert decoder.embed_positions.weight.shape[1] == hidden_size
assert decoder.embed_positions.weight.shape == (
    max_positions + 2,
    hidden_size
)
print("All Component 1 probes passed.")

All Component 1 probes passed.


# Component 2: Attention Projections

## 1. Algorithmic Specification

The attention projection stage converts the input hidden representation into separate query, key, and value representations.

### Input
- Hidden states from the previous stage:

\[
H \in \mathbb{R}^{B \times T \times d_{\text{model}}}
\]

### Output
- Query matrix \(Q\)
- Key matrix \(K\)
- Value matrix \(V\)

Before splitting into attention heads:

\[
Q, K, V \in \mathbb{R}^{B \times T \times d_{\text{model}}}
\]

After splitting into heads:

\[
Q, K, V \in \mathbb{R}^{B \times H \times T \times d_{\text{head}}}
\]

where

\[
d_{\text{head}} = \frac{d_{\text{model}}}{H}
\]

### Learned Parameters
The attention layer contains three learned projection matrices:

\[
W_Q \in \mathbb{R}^{d_{\text{model}} \times d_{\text{model}}}
\]

\[
W_K \in \mathbb{R}^{d_{\text{model}} \times d_{\text{model}}}
\]

\[
W_V \in \mathbb{R}^{d_{\text{model}} \times d_{\text{model}}}
\]

and corresponding bias vectors:

\[
b_Q, b_K, b_V \in \mathbb{R}^{d_{\text{model}}}
\]

### Relevant Model Settings
- \(d_{\text{model}}\) = hidden size
- \(H\) = number of attention heads
- \(d_{\text{head}} = d_{\text{model}} / H\)
- \(T\) = sequence length
- \(B\) = batch size

## 2. Predicted Dimensions

For OPT-125M:

- Batch size: `B = 1`
- Sequence length: `T = 8`
- Hidden size: `d_model = 768`
- Number of attention heads: `H = 12`

Therefore:

`d_head = d_model / H = 768 / 12 = 64`

Predicted dimensions:

- Input hidden states: `(1, 8, 768)`

- Query projection weight matrix: `(768, 768)`

- Key projection weight matrix: `(768, 768)`

- Value projection weight matrix: `(768, 768)`

- Query bias: `(768)`

- Key bias: `(768)`

- Value bias: `(768)`

- Projected `Q`, `K`, and `V` before splitting into heads: `(1, 8, 768)`

- `Q`, `K`, and `V` after splitting into 12 heads: `(1, 12, 8, 64)`

Each token's 768-dimensional hidden representation is divided across 12 attention heads, giving each head a 64-dimensional representation.

## 3. Formal Pseudocode

### Input

- Hidden states `H` with shape `(B, T, d_model)`

### Parameters

- Query projection matrix `W_Q` with shape `(d_model, d_model)`
- Key projection matrix `W_K` with shape `(d_model, d_model)`
- Value projection matrix `W_V` with shape `(d_model, d_model)`
- Bias vectors `b_Q`, `b_K`, and `b_V` with shape `(d_model)`

### Procedure

1. Compute the query projection:

`Q <- H W_Q + b_Q`

2. Compute the key projection:

`K <- H W_K + b_K`

3. Compute the value projection:

`V <- H W_V + b_V`

4. Split each projection into `H` attention heads:

`Q <- reshape(Q, B, T, H, d_head)`

`K <- reshape(K, B, T, H, d_head)`

`V <- reshape(V, B, T, H, d_head)`

5. Rearrange the head dimension before the sequence dimension:

`Q, K, V` have shape `(B, H, T, d_head)`

### Output

`Q, K, V` with shape `(B, H, T, d_head)`

In [10]:
layer0 = decoder.layers[0]
attention = layer0.self_attn

print("Query projection:")
print(attention.q_proj)

print("\nKey projection:")
print(attention.k_proj)

print("\nValue projection:")
print(attention.v_proj)

print("\nQ weight shape:", attention.q_proj.weight.shape)
print("K weight shape:", attention.k_proj.weight.shape)
print("V weight shape:", attention.v_proj.weight.shape)

print("\nQ bias shape:", attention.q_proj.bias.shape)
print("K bias shape:", attention.k_proj.bias.shape)
print("V bias shape:", attention.v_proj.bias.shape)

print("\nNumber of heads:", attention.num_heads)
print("Head dimension:", attention.head_dim)

Query projection:
Linear(in_features=768, out_features=768, bias=True)

Key projection:
Linear(in_features=768, out_features=768, bias=True)

Value projection:
Linear(in_features=768, out_features=768, bias=True)

Q weight shape: torch.Size([768, 768])
K weight shape: torch.Size([768, 768])
V weight shape: torch.Size([768, 768])

Q bias shape: torch.Size([768])
K bias shape: torch.Size([768])
V bias shape: torch.Size([768])

Number of heads: 12
Head dimension: 64


In [11]:
q = attention.q_proj(hidden_states)
k = attention.k_proj(hidden_states)
v = attention.v_proj(hidden_states)

batch_size, seq_len, hidden_size = q.shape
num_heads = attention.num_heads
head_dim = attention.head_dim

q_heads = q.view(batch_size, seq_len, num_heads, head_dim).transpose(1, 2)
k_heads = k.view(batch_size, seq_len, num_heads, head_dim).transpose(1, 2)
v_heads = v.view(batch_size, seq_len, num_heads, head_dim).transpose(1, 2)

print("Q before head split:", q.shape)
print("K before head split:", k.shape)
print("V before head split:", v.shape)

print("\nQ after head split:", q_heads.shape)
print("K after head split:", k_heads.shape)
print("V after head split:", v_heads.shape)

Q before head split: torch.Size([1, 8, 768])
K before head split: torch.Size([1, 8, 768])
V before head split: torch.Size([1, 8, 768])

Q after head split: torch.Size([1, 12, 8, 64])
K after head split: torch.Size([1, 12, 8, 64])
V after head split: torch.Size([1, 12, 8, 64])


## 4. Observed Results and Corrections

The observed query, key, and value projection matrices all had dimensions:

`(768, 768)`

Their corresponding bias vectors each had dimensions:

`(768)`

For the test input, the query, key, and value tensors before splitting into attention heads had dimensions:

`(1, 8, 768)`

After splitting the hidden dimension across 12 attention heads, the tensors had dimensions:

`(1, 12, 8, 64)`

This matches the expected relationship:

`d_head = d_model / H = 768 / 12 = 64`

The observed tensor dimensions matched the predicted dimensions, so no correction was needed for this component.

In [12]:
# Component 2 probes

batch_size, seq_len = inputs["input_ids"].shape
hidden_size = config.hidden_size
num_heads = config.num_attention_heads
head_dim = hidden_size // num_heads

assert hidden_size % num_heads == 0
assert head_dim == attention.head_dim

assert attention.q_proj.weight.shape == (hidden_size, hidden_size)
assert attention.k_proj.weight.shape == (hidden_size, hidden_size)
assert attention.v_proj.weight.shape == (hidden_size, hidden_size)

assert attention.q_proj.bias.shape == (hidden_size,)
assert attention.k_proj.bias.shape == (hidden_size,)
assert attention.v_proj.bias.shape == (hidden_size,)

assert q.shape == (batch_size, seq_len, hidden_size)
assert k.shape == (batch_size, seq_len, hidden_size)
assert v.shape == (batch_size, seq_len, hidden_size)

assert q_heads.shape == (batch_size, num_heads, seq_len, head_dim)
assert k_heads.shape == (batch_size, num_heads, seq_len, head_dim)
assert v_heads.shape == (batch_size, num_heads, seq_len, head_dim)

print("All Component 2 probes passed.")

All Component 2 probes passed.


# Component 3: Attention Calculation

## 1. Algorithmic Specification

The attention calculation uses the projected query, key, and value tensors to determine how strongly each token should attend to other token positions.

### Input

- Query tensor `Q` with shape `(B, H, T, d_head)`
- Key tensor `K` with shape `(B, H, T, d_head)`
- Value tensor `V` with shape `(B, H, T, d_head)`

### Output

- Attention score matrix for each head
- Normalized attention weights
- Weighted combination of value vectors
- Final attention output after combining all heads and applying the output projection

### Learned Parameters

The attention score, masking, and softmax operations do not introduce new learned parameters.

The final attention output projection contains:

- Output projection matrix `W_O` with shape `(d_model, d_model)`
- Output projection bias `b_O` with shape `(d_model)`

### Relevant Model Settings

- `B` = batch size
- `T` = sequence length
- `H` = number of attention heads
- `d_head` = dimension of one attention head
- `d_model` = hidden size

## 2. Predicted Dimensions

For OPT-125M:

- Batch size: `B = 1`
- Sequence length: `T = 8`
- Number of attention heads: `H = 12`
- Head dimension: `d_head = 64`
- Hidden size: `d_model = 768`

The query and key tensors have dimensions:

`(1, 12, 8, 64)`

To calculate attention scores, each query vector is compared with every key vector in the same attention head.

Therefore, the attention score tensor should have dimensions:

`(1, 12, 8, 8)`

The causal mask must be compatible with the attention score matrix and prevent each token from attending to future token positions.

After applying the causal mask and softmax, the attention weights should remain:

`(1, 12, 8, 8)`

The attention weights are then multiplied by the value vectors:

`(1, 12, 8, 8) x (1, 12, 8, 64)`

This produces a per-head attention output with dimensions:

`(1, 12, 8, 64)`

After the 12 attention heads are combined, the tensor should return to:

`(1, 8, 768)`

The output projection weight matrix is predicted to have dimensions:

`(768, 768)`

The output projection bias is predicted to have dimensions:

`(768)`

The final output of the attention sublayer should have dimensions:

`(1, 8, 768)`

## 3. Formal Pseudocode

### Input

- Query tensor `Q` with shape `(B, H, T, d_head)`
- Key tensor `K` with shape `(B, H, T, d_head)`
- Value tensor `V` with shape `(B, H, T, d_head)`

### Parameters

- Output projection matrix `W_O` with shape `(d_model, d_model)`
- Output projection bias `b_O` with shape `(d_model)`

### Procedure

1. Compute the scaled attention scores:

`S <- (Q K^T) / sqrt(d_head)`

The attention score tensor `S` has shape:

`(B, H, T, T)`

2. Apply the causal mask so that token position `i` cannot attend to future positions where `j > i`:

`S <- S + M_causal`

3. Normalize the attention scores using softmax:

`A <- softmax(S)`

4. Compute the weighted combination of value vectors:

`C <- A V`

The resulting tensor `C` has shape:

`(B, H, T, d_head)`

5. Combine the attention heads:

`C_combined <- reshape(C, B, T, d_model)`

6. Apply the output projection:

`Y <- C_combined W_O + b_O`

### Output

`Y` with shape `(B, T, d_model)`

In [13]:
import math
import torch

# Scale query vectors
scaled_q = q_heads / math.sqrt(head_dim)

# Attention scores
attention_scores = torch.matmul(
    scaled_q,
    k_heads.transpose(-2, -1)
)

# Causal mask
causal_mask = torch.triu(
    torch.ones(seq_len, seq_len, dtype=torch.bool),
    diagonal=1
)

masked_scores = attention_scores.masked_fill(
    causal_mask,
    float("-inf")
)

# Softmax normalization
attention_weights = torch.softmax(masked_scores, dim=-1)

# Weighted sum of values
attention_output_heads = torch.matmul(
    attention_weights,
    v_heads
)

# Combine heads
attention_output_combined = (
    attention_output_heads
    .transpose(1, 2)
    .contiguous()
    .view(batch_size, seq_len, hidden_size)
)

# Output projection
attention_output = attention.out_proj(attention_output_combined)

print("Attention scores:", attention_scores.shape)
print("Causal mask:", causal_mask.shape)
print("Attention weights:", attention_weights.shape)
print("Per-head attention output:", attention_output_heads.shape)
print("Combined attention output:", attention_output_combined.shape)
print("Output projection weight:", attention.out_proj.weight.shape)
print("Output projection bias:", attention.out_proj.bias.shape)
print("Final attention output:", attention_output.shape)

Attention scores: torch.Size([1, 12, 8, 8])
Causal mask: torch.Size([8, 8])
Attention weights: torch.Size([1, 12, 8, 8])
Per-head attention output: torch.Size([1, 12, 8, 64])
Combined attention output: torch.Size([1, 8, 768])
Output projection weight: torch.Size([768, 768])
Output projection bias: torch.Size([768])
Final attention output: torch.Size([1, 8, 768])


## 4. Observed Results and Corrections

The observed attention score tensor had dimensions:

\[
(1, 12, 8, 8)
\]

This matches the expectation that each of the 8 query positions is compared with all 8 key positions independently within each of the 12 attention heads.

The causal mask had dimensions:

\[
(8, 8)
\]

The mask is broadcast across the batch and attention-head dimensions when applied to the attention scores.

After applying the causal mask and softmax, the attention weight tensor remained:

\[
(1, 12, 8, 8)
\]

The weighted combination of the value vectors produced a per-head output with dimensions:

\[
(1, 12, 8, 64)
\]

After recombining the 12 heads, the tensor returned to:

\[
(1, 8, 768)
\]

The output projection had weight dimensions:

\[
(768, 768)
\]

and bias dimension:

\[
(768)
\]

The final attention output had dimensions:

\[
(1, 8, 768)
\]

No major correction was needed in this section because the observed tensor dimensions matched the predictions.

In [20]:
# Component 3 probes

batch_size, seq_len = inputs["input_ids"].shape
hidden_size = config.hidden_size
num_heads = config.num_attention_heads
head_dim = hidden_size // num_heads

assert attention_scores.shape == (
    batch_size,
    num_heads,
    seq_len,
    seq_len
)

assert causal_mask.shape == (
    seq_len,
    seq_len
)

assert attention_weights.shape == (
    batch_size,
    num_heads,
    seq_len,
    seq_len
)

assert attention_output_heads.shape == (
    batch_size,
    num_heads,
    seq_len,
    head_dim
)

assert attention_output_combined.shape == (
    batch_size,
    seq_len,
    hidden_size
)

assert attention.out_proj.weight.shape == (
    hidden_size,
    hidden_size
)

assert attention.out_proj.bias.shape == (
    hidden_size,
)

assert attention_output.shape == (
    batch_size,
    seq_len,
    hidden_size
)

# Each attention row should sum to approximately 1
row_sums = attention_weights.sum(dim=-1)

assert torch.allclose(
    row_sums,
    torch.ones_like(row_sums),
    atol=1e-5
)

# Future token positions should receive zero attention weight
future_positions = torch.triu(
    torch.ones(seq_len, seq_len, dtype=torch.bool),
    diagonal=1
)

future_attention = attention_weights.masked_select(
    future_positions.unsqueeze(0).unsqueeze(0)
)

assert torch.allclose(
    future_attention,
    torch.zeros_like(future_attention),
    atol=1e-7
)
print("All Component 3 probes passed.")

All Component 3 probes passed.


# Component 4: Transformer Decoder Layer

## 1. Algorithmic Specification

A single OPT decoder layer applies:

1. Layer normalization
2. Self-attention
3. An attention residual connection
4. A feed-forward network
5. A second residual connection

For OPT-125M, the feed-forward network expands the hidden representation from `d_model` to `d_ff`, applies a non-linear activation, then projects it back to `d_model`.

### Input

- Hidden states `X` with shape `(B, T, d_model)`

### Output

- Decoder-layer output `Y` with shape `(B, T, d_model)`

### Learned Parameters

The layer contains two sets of LayerNorm parameters:

- Attention LayerNorm scale `gamma_attn` with shape `(d_model)`
- Attention LayerNorm bias `beta_attn` with shape `(d_model)`
- Feed-forward LayerNorm scale `gamma_ffn` with shape `(d_model)`
- Feed-forward LayerNorm bias `beta_ffn` with shape `(d_model)`

The feed-forward network contains:

- First projection matrix `W_1` with shape `(d_model, d_ff)`
- First projection bias `b_1` with shape `(d_ff)`
- Second projection matrix `W_2` with shape `(d_ff, d_model)`
- Second projection bias `b_2` with shape `(d_model)`

The self-attention parameters are the query, key, value, and output projection parameters investigated in Components 2 and 3.

### Relevant Model Settings

- `d_model` = hidden size
- `d_ff` = feed-forward dimension
- `T` = sequence length
- `B` = batch size

## 2. Predicted Dimensions

For OPT-125M:

- Batch size: `B = 1`
- Sequence length: `T = 8`
- Hidden size: `d_model = 768`
- Feed-forward dimension: `d_ff = 3072`

Predicted dimensions through the decoder layer:

- Input hidden states: `(1, 8, 768)`

- After attention LayerNorm: `(1, 8, 768)`

- Self-attention output: `(1, 8, 768)`

- After the attention residual connection: `(1, 8, 768)`

- Feed-forward first projection: `(1, 8, 3072)`

- After the activation function: `(1, 8, 3072)`

- Feed-forward second projection: `(1, 8, 768)`

- After the feed-forward residual connection: `(1, 8, 768)`

Predicted feed-forward parameter dimensions in PyTorch storage order:

- `W_1`: `(3072, 768)`
- `b_1`: `(3072)`
- `W_2`: `(768, 3072)`
- `b_2`: `(768)`

PyTorch stores Linear-layer weights in `(out features, in features)` order. Therefore, the stored weight dimensions appear transposed relative to the row-major mathematical notation.

## 3. Formal Pseudocode

### Input

- Hidden states `X` with shape `(B, T, d_model)`

### Procedure

1. Save the residual input:

`R_1 <- X`

2. Apply attention LayerNorm:

`X_1 <- LayerNorm(X)`

3. Apply self-attention:

`A <- SelfAttention(X_1)`

4. Add the attention residual connection:

`X_2 <- R_1 + A`

5. Save the second residual input:

`R_2 <- X_2`

6. Apply feed-forward LayerNorm:

`X_3 <- LayerNorm(X_2)`

7. Apply the first feed-forward projection:

`F_1 <- X_3 W_1 + b_1`

8. Apply the activation function:

`F_2 <- Activation(F_1)`

9. Project back to the model hidden size:

`F_3 <- F_2 W_2 + b_2`

10. Add the feed-forward residual connection:

`Y <- R_2 + F_3`

### Output

- Decoder-layer output `Y` with shape `(B, T, d_model)`

In [15]:
layer0 = decoder.layers[0]

print("Self-attention LayerNorm:", layer0.self_attn_layer_norm)
print("Final LayerNorm:", layer0.final_layer_norm)

print("\nFFN first layer:", layer0.fc1)
print("FFN second layer:", layer0.fc2)

print("\nfc1 weight:", layer0.fc1.weight.shape)
print("fc1 bias:", layer0.fc1.bias.shape)

print("\nfc2 weight:", layer0.fc2.weight.shape)
print("fc2 bias:", layer0.fc2.bias.shape)

print("\nAttention LayerNorm weight:",
      layer0.self_attn_layer_norm.weight.shape)

print("Final LayerNorm weight:",
      layer0.final_layer_norm.weight.shape)

Self-attention LayerNorm: LayerNorm((768,), eps=1e-05, elementwise_affine=True)
Final LayerNorm: LayerNorm((768,), eps=1e-05, elementwise_affine=True)

FFN first layer: Linear(in_features=768, out_features=3072, bias=True)
FFN second layer: Linear(in_features=3072, out_features=768, bias=True)

fc1 weight: torch.Size([3072, 768])
fc1 bias: torch.Size([3072])

fc2 weight: torch.Size([768, 3072])
fc2 bias: torch.Size([768])

Attention LayerNorm weight: torch.Size([768])
Final LayerNorm weight: torch.Size([768])


In [16]:
# Trace Component 4 through the first decoder layer

x = hidden_states

# 1. Attention LayerNorm
x_attn_norm = layer0.self_attn_layer_norm(x)

# 2. Self-attention
attn_result = layer0.self_attn(
    hidden_states=x_attn_norm,
    attention_mask=None,
)[0]

# 3. Attention residual
x_after_attn = x + attn_result

# 4. FFN LayerNorm
x_ffn_norm = layer0.final_layer_norm(x_after_attn)

# 5. First FFN projection
ffn_hidden = layer0.fc1(x_ffn_norm)

# 6. Activation
ffn_activated = layer0.activation_fn(ffn_hidden)

# 7. Second FFN projection
ffn_output = layer0.fc2(ffn_activated)

# 8. FFN residual
layer_output = x_after_attn + ffn_output

print("Input:", x.shape)
print("After attention LayerNorm:", x_attn_norm.shape)
print("Attention output:", attn_result.shape)
print("After attention residual:", x_after_attn.shape)
print("After FFN LayerNorm:", x_ffn_norm.shape)
print("After fc1:", ffn_hidden.shape)
print("After activation:", ffn_activated.shape)
print("After fc2:", ffn_output.shape)
print("Final layer output:", layer_output.shape)

Input: torch.Size([1, 8, 768])
After attention LayerNorm: torch.Size([1, 8, 768])
Attention output: torch.Size([1, 8, 768])
After attention residual: torch.Size([1, 8, 768])
After FFN LayerNorm: torch.Size([1, 8, 768])
After fc1: torch.Size([1, 8, 3072])
After activation: torch.Size([1, 8, 3072])
After fc2: torch.Size([1, 8, 768])
Final layer output: torch.Size([1, 8, 768])


## 4. Observed Results and Corrections

The input to the first decoder layer had dimensions:

`(1, 8, 768)`

Layer normalization preserved the tensor dimensions:

`(1, 8, 768)`

The self-attention output also had dimensions:

`(1, 8, 768)`

After adding the attention residual connection, the tensor remained:

`(1, 8, 768)`

The feed-forward network then expanded the hidden dimension:

`(1, 8, 768) -> (1, 8, 3072)`

The activation function preserved the feed-forward shape:

`(1, 8, 3072)`

The second feed-forward projection reduced the representation back to the model hidden size:

`(1, 8, 3072) -> (1, 8, 768)`

After adding the feed-forward residual connection, the final decoder-layer output had dimensions:

`(1, 8, 768)`

The observed results matched the predicted dimensions, so no major correction was required for this component.

In [17]:
# Component 4 probes

batch_size, seq_len = inputs["input_ids"].shape
hidden_size = config.hidden_size
ffn_dim = config.ffn_dim

assert x.shape == (
    batch_size,
    seq_len,
    hidden_size
)

assert x_attn_norm.shape == x.shape
assert attn_result.shape == x.shape
assert x_after_attn.shape == x.shape

assert x_ffn_norm.shape == x.shape

assert ffn_hidden.shape == (
    batch_size,
    seq_len,
    ffn_dim
)

assert ffn_activated.shape == (
    batch_size,
    seq_len,
    ffn_dim
)

assert ffn_output.shape == (
    batch_size,
    seq_len,
    hidden_size
)

assert layer_output.shape == (
    batch_size,
    seq_len,
    hidden_size
)

assert layer0.fc1.weight.shape == (
    ffn_dim,
    hidden_size
)

assert layer0.fc2.weight.shape == (
    hidden_size,
    ffn_dim
)

print("All Component 4 probes passed.")

All Component 4 probes passed.


# Component 5: Decoder and Language Model Output

## 1. Algorithmic Specification

The full decoder repeatedly applies the Transformer decoder layer across all `L` layers. After the final decoder layer, OPT applies a final LayerNorm and then the language-model head converts each token representation into vocabulary scores.

### Input

- Initial hidden states `H^(0)` with shape `(B, T, d_model)`

### Repeated Decoder Layers

For each decoder layer `l = 1, 2, ..., L`:

`H^(l) <- DecoderLayer_l(H^(l-1))`

Each decoder layer preserves the hidden-state dimensions:

`H^(l)` has shape `(B, T, d_model)`

### Final Normalization

After the last decoder layer:

`H_final <- LayerNorm(H^(L))`

The final hidden representation remains:

`H_final` with shape `(B, T, d_model)`

### Language Model Head

The language-model head maps every token's final hidden representation to one score for every token in the vocabulary.

For OPT-125M, the language-model head is a linear transformation without a bias term:

`Z <- H_final W_LM`

Using row-major mathematical notation, the language-model weight matrix `W_LM` has shape:

`(d_model, V)`

PyTorch stores this Linear-layer weight matrix in:

`(V, d_model)`

order.

The resulting values are the vocabulary logits.

### Output

- Vocabulary logits `Z` with shape `(B, T, V)`

## 2. Predicted Dimensions

For OPT-125M:

- Batch size: B = 1
- Sequence length: T = 8
- Number of decoder layers: L = 12
- Hidden size: d_model = 768
- Vocabulary size: V = 50,272

The input to the decoder has dimensions:

`(B, T, d_model) = (1, 8, 768)`

Each of the 12 decoder layers preserves the hidden-state dimensions:

`(1, 8, 768)`

Therefore, after all 12 decoder layers, the hidden representation is still:

`(1, 8, 768)`

The final decoder LayerNorm operates across the hidden dimension.

Predicted final LayerNorm weight dimensions:

`(d_model) = (768)`

Predicted final LayerNorm bias dimensions:

`(d_model) = (768)`

The final LayerNorm preserves the hidden-state dimensions:

`(1, 8, 768)`

The language-model head maps each 768-dimensional hidden vector to one score for every token in the vocabulary.

In row-major mathematical notation, the language-model weight matrix has dimensions:

`(d_model, V) = (768, 50,272)`

PyTorch stores Linear-layer weights in `(out features, in features)` order, so the stored LM-head weight matrix is predicted to have dimensions:

`(V, d_model) = (50,272, 768)`

OPT-125M's language-model head does not use a bias term.

The final vocabulary logits are predicted to have dimensions:

`(B, T, V) = (1, 8, 50,272)`

This means that the model produces 50,272 vocabulary scores for each of the 8 token positions.

## 3. Formal Pseudocode

### Input

- Initial hidden states `H^(0)` with shape `(B, T, d_model)`

### Procedure

1. Repeatedly apply the decoder layers:

`for l = 1 to L:`

`H^(l) <- DecoderLayer_l(H^(l-1))`

2. Apply the final decoder normalization:

`H_final <- LayerNorm(H^(L))`

3. Apply the language-model head:

`Z <- H_final W_LM`

### Output

- Vocabulary logits `Z` with shape `(B, T, V)`

`Z` contains one vocabulary score, or logit, for every token in the vocabulary at every sequence position.

In [21]:
with torch.no_grad():
    outputs = model(
        **inputs,
        output_hidden_states=True,
        return_dict=True
    )

print("Number of decoder layers:", config.num_hidden_layers)
print("Number of hidden-state tensors:", len(outputs.hidden_states))

for i, h in enumerate(outputs.hidden_states):
    print(f"Hidden state {i}:", h.shape)

print("\nFinal decoder LayerNorm:")
print(decoder.final_layer_norm)

print("\nFinal LayerNorm weight shape:")
print(decoder.final_layer_norm.weight.shape)

print("\nFinal LayerNorm bias shape:")
print(decoder.final_layer_norm.bias.shape)

print("\nLM head:")
print(model.lm_head)

print("\nLM head weight:")
print(model.lm_head.weight.shape)

print("\nLM head bias:")
print(model.lm_head.bias)

print("\nFinal logits:")
print(outputs.logits.shape)

Number of decoder layers: 12
Number of hidden-state tensors: 13
Hidden state 0: torch.Size([1, 8, 768])
Hidden state 1: torch.Size([1, 8, 768])
Hidden state 2: torch.Size([1, 8, 768])
Hidden state 3: torch.Size([1, 8, 768])
Hidden state 4: torch.Size([1, 8, 768])
Hidden state 5: torch.Size([1, 8, 768])
Hidden state 6: torch.Size([1, 8, 768])
Hidden state 7: torch.Size([1, 8, 768])
Hidden state 8: torch.Size([1, 8, 768])
Hidden state 9: torch.Size([1, 8, 768])
Hidden state 10: torch.Size([1, 8, 768])
Hidden state 11: torch.Size([1, 8, 768])
Hidden state 12: torch.Size([1, 8, 768])

Final decoder LayerNorm:
LayerNorm((768,), eps=1e-05, elementwise_affine=True)

Final LayerNorm weight shape:
torch.Size([768])

Final LayerNorm bias shape:
torch.Size([768])

LM head:
Linear(in_features=768, out_features=50272, bias=False)

LM head weight:
torch.Size([50272, 768])

LM head bias:
None

Final logits:
torch.Size([1, 8, 50272])


## 4. Observed Results and Corrections

OPT-125M contains 12 decoder layers.

When hidden states were requested from Hugging Face, the model returned 13 hidden-state tensors:

- One tensor for the initial hidden representation
- One tensor after each of the 12 decoder layers

Each hidden-state tensor had dimensions:

`(1, 8, 768)`

This confirms that each decoder layer preserves the sequence length and hidden size.

After the repeated decoder layers, OPT applies a final LayerNorm over the hidden dimension. Its learned weight and bias vectors each had dimensions:

`(768)`

The final LayerNorm preserves the hidden-state dimensions:

`(1, 8, 768)`

The language-model head maps the hidden representation from 768 dimensions to the vocabulary size of 50,272.

Its stored weight matrix had dimensions:

`(50,272, 768)`

The language-model head does not use a bias term.

The final logits had dimensions:

`(1, 8, 50,272)`

This means that for each of the 8 token positions, the model produced one score for every token in the 50,272-token vocabulary.

One detail clarified by observation was that Hugging Face returns `L + 1` hidden-state tensors when `output_hidden_states=True`. For OPT-125M, `L = 12`, so 13 tensors are returned because the initial hidden representation is included along with the output of each decoder layer.

Another correction was that the language-model head does not use a bias vector. The final computation is therefore:

`Z = H_final W_LM`

rather than adding a language-model bias term.

In [22]:
# Component 5 probes

batch_size, seq_len = inputs["input_ids"].shape
hidden_size = config.hidden_size
num_layers = config.num_hidden_layers
vocab_size = config.vocab_size

# Hugging Face returns the initial hidden state plus
# one output for each decoder layer
assert len(outputs.hidden_states) == num_layers + 1

# Every decoder layer preserves the hidden-state dimensions
for h in outputs.hidden_states:
    assert h.shape == (
        batch_size,
        seq_len,
        hidden_size
    )

# Final LayerNorm operates over the hidden dimension
assert decoder.final_layer_norm.weight.shape == (
    hidden_size,
)

assert decoder.final_layer_norm.bias.shape == (
    hidden_size,
)

# Language-model head maps hidden size to vocabulary size
assert model.lm_head.in_features == hidden_size
assert model.lm_head.out_features == vocab_size

assert model.lm_head.weight.shape == (
    vocab_size,
    hidden_size
)

# OPT's language-model head does not use a bias
assert model.lm_head.bias is None

# The model produces one vocabulary score for every token position
assert outputs.logits.shape == (
    batch_size,
    seq_len,
    vocab_size
)

print("All Component 5 probes passed.")

All Component 5 probes passed.


# Assignment Summary

This investigation traced Meta's OPT-125M decoder-only Transformer from its input representation through the final vocabulary logits. The goal was to connect the formal description of Transformer operations with Hugging Face's PyTorch implementation and the tensor shapes observed during execution.

The model configuration showed that OPT-125M uses a hidden size of 768, 12 attention heads, 12 decoder layers, a feed-forward dimension of 3072, a maximum sequence length of 2048, and a vocabulary size of 50,272.

For the test sequence containing 8 tokens, the initial token and positional embeddings produced hidden states with shape `(1, 8, 768)`. OPT's positional embedding table contained 2050 rows rather than 2048 because the implementation uses a positional offset of 2.

The query, key, and value projections each preserved the 768-dimensional hidden representation before splitting it across 12 attention heads. After the split, each tensor had shape `(1, 12, 8, 64)`, confirming that each head operates on 64 dimensions.

During the attention calculation, the query and key tensors produced attention score matrices with shape `(1, 12, 8, 8)`. The causal mask prevented tokens from attending to future positions, and softmax converted the scores into normalized attention weights. Multiplying these weights by the value tensors produced per-head outputs with shape `(1, 12, 8, 64)`. Combining the heads returned the representation to `(1, 8, 768)`.

Within each Transformer decoder layer, LayerNorm and residual connections preserved the hidden-state dimensions. The feed-forward network temporarily expanded the representation from 768 dimensions to 3072 dimensions before projecting it back to 768. The final output of each decoder layer therefore remained `(1, 8, 768)`.

Across the full decoder, all 12 Transformer layers preserved the sequence length and hidden size. Hugging Face returned 13 hidden-state tensors because it includes the initial hidden representation along with the output of each of the 12 decoder layers. After the final LayerNorm, the language-model head projected each 768-dimensional hidden vector into 50,272 vocabulary scores. The final logits therefore had shape `(1, 8, 50,272)`.

The probes confirmed the expected dimensional relationships throughout the model. They also verified behavioral properties such as attention weights summing to approximately 1 and future token positions receiving zero attention after causal masking.

Overall, the investigation showed that the formal Transformer operations, Hugging Face's OPT implementation, and the observed runtime tensors were consistent with one another. The main corrections made during the investigation involved OPT's positional embedding offset, the additional initial hidden-state tensor returned by Hugging Face, and the fact that the language-model head does not use a bias term.